# VAEs & GANs

Here we'll try to implement the variational autoencoders and the generative-adversarial networks.

## Importing

first let's do our imports and our data

In [1]:
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import einops
import torch as t
import torchinfo
import wandb
from datasets import load_dataset
from einops.layers.torch import Rearrange
from jaxtyping import Float
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm import tqdm

## to add the arena-helpers (tests, utils)
sys.path.insert(0, '../arena_helpers')
sys.path.insert(0, '..')

import arena_helpers.tests as tests
import arena_helpers.utils as utils
from plotly_utils import imshow

In [2]:
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
device

device(type='cuda')

## Downloading our Data

Here we'll download the data we need for our exercises today. Then load it and do the needed transforms.

In [3]:
current_dir = Path.cwd()
print(current_dir)

/home/youssef/projects/AI-safety/arena-3.0/prereqs/vaes-gans


In [4]:
celeb_data_dir = current_dir/ "data/celeba"
celeb_image_dir = celeb_data_dir / "img_align_celeba"

os.makedirs(celeb_image_dir, exist_ok=True)

if len(list(celeb_image_dir.glob("*.jpg"))) > 0:
    print("Dataset already loaded.")
else:
    dataset = load_dataset("nielsr/CelebA-faces")
    print("Dataset loaded.")

    for idx, item in tqdm(enumerate(dataset["train"]), total=len(dataset["train"]), desc="Saving imgs...", ascii=True):
        # The image is already a JpegImageFile, so we can directly save it
        item["image"].save(celeb_image_dir / f"{idx:06}.jpg")

    print("All images have been saved.")

data/train-00000-of-00003.parquet:   0%|          | 0.00/462M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

Here we'll load the MNIST dataset that can be used instead of the celebrity dataset for the VAE.

In [13]:
from datasets import load_dataset
import os

mnist_data_dir =   Path("data/mnist")
mnist_image_dir = mnist_data_dir / "img_mnist"
os.makedirs(mnist_image_dir, exist_ok=True)

if len(list(mnist_image_dir.glob("*.jpg"))) > 0:
    print("Dataset already loaded.")
else:
    dataset = load_dataset("ylecun/mnist")
    print("Dataset loaded.")
    for split in ["train", "test"]:
        split_dir = mnist_image_dir / split
        os.makedirs(split_dir, exist_ok=True)
        for idx, item in tqdm(enumerate(dataset[split]), total=len(dataset[split]), desc=f"Saving {split} imgs...", ascii=True):
            item["image"].save(split_dir / f"{idx:06}.jpg")
    print("All images have been saved.")

README.md: 0.00B [00:00, ?B/s]

mnist/train-00000-of-00001.parquet:   0%|          | 0.00/15.6M [00:00<?, ?B/s]

mnist/test-00000-of-00001.parquet:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset loaded.


Saving test imgs...: 100%|##########| 10000/10000 [00:02<00:00, 3876.38it/s]

All images have been saved.


To apply the needed transforms, here is the code below that uses `torchvision.datasets`

In [16]:
def get_dataset(dataset: Literal["MNIST", "CELEB"], train: bool = True) -> Dataset:
    assert dataset in ["MNIST", "CELEB"]

    if dataset == "CELEB":
        image_size = 64
        assert train, "CelebA dataset only has a training set"
        transform = transforms.Compose(
            [
                transforms.Resize(image_size),
                transforms.CenterCrop(image_size),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ]
        )
        trainset = datasets.ImageFolder(root= "/data/celeba", transform=transform)

    elif dataset == "MNIST":
        img_size = 28
        transform = transforms.Compose(
            [
                transforms.Resize(img_size),
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )
        trainset = datasets.MNIST(
            root= "./data/mnist",
            transform=transform,
            download=True,
            train=train,
        )

    return trainset

The code below will be used for visualization

In [17]:
def display_data(x: Tensor, nrows: int, title: str):
    """Displays a batch of data, using plotly."""
    ncols = x.shape[0] // nrows
    # Reshape into the right shape for plotting (make it 2D if image is monochrome)
    y = einops.rearrange(x, "(b1 b2) c h w -> (b1 h) (b2 w) c", b1=nrows).squeeze()
    # Normalize in the 0-1 range, then map to integer type
    y = (y - y.min()) / (y.max() - y.min())
    y = (y * 255).to(dtype=t.uint8)
    # Display data
    imshow(
        y,
        binary_string=(y.ndim == 2),
        height=50 * (nrows + 4),
        width=50 * (ncols + 5),
        title=f"{title}<br>single input shape = {x[0].shape}",
    )


trainset_mnist = get_dataset("MNIST")
#trainset_celeb = get_dataset("CELEB")

# Display MNIST
x = next(iter(DataLoader(trainset_mnist, batch_size=25)))[0]
display_data(x, nrows=5, title="MNIST data")

# Display CelebA
#x = next(iter(DataLoader(trainset_celeb, batch_size=25)))[0]
#display_data(x, nrows=5, title="CelebA data")

100%|██████████| 9.91M/9.91M [00:19<00:00, 499kB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 76.5kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.17MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.44MB/s]


Next we'll hold out some data to not be used during training. 

In [18]:
testset = get_dataset("MNIST", train=False)
HOLDOUT_DATA = dict()
for data, target in DataLoader(testset, batch_size=1):
    if target.item() not in HOLDOUT_DATA:
        HOLDOUT_DATA[target.item()] = data.squeeze()
        if len(HOLDOUT_DATA) == 10:
            break
HOLDOUT_DATA = t.stack([HOLDOUT_DATA[i] for i in range(10)]).to(dtype=t.float, device=device).unsqueeze(1)

display_data(HOLDOUT_DATA, nrows=1, title="MNIST holdout data")

# Autoencoders

First thing we'll play with are the autoencoders. These systems learn a compressed representation of the input, that's why we first compress the input then reconstruct it, this may seem pointless but the whole point is for the system to learn the weights to be able to construct an image given a class.

Autoencoders contain:
- **encoder:** learns to compress the output into a latent space of lower-dimensionality than the original image.
- **decoder:** learns to uncompress the encoder's output back into a faithful representation oof the image.

In this simple exercise, we'll have two convolution blocks, followed by two fully connected layers. 

The decoder will be the mirror image but convolutions will be transposed convolutions.

In [31]:
# Go up/down directories relative to current file
path = Path.cwd().parent / "cnn-resnets"
print(path.exists())

True


In [37]:
# Importing all modules you'll need, from previous solutions 
sys.path.append(str(Path.cwd().parent / "cnn-resnets"))
from cnn_resnets import BatchNorm2d, Conv2d, Linear, ReLU, Sequential


class Autoencoder(nn.Module):
    def __init__(self, latent_dim_size: int, hidden_dim_size: int):
        """Creates the encoder & decoder modules."""
        super().__init__()
        self.in_channels1 = 1568 # c*h*w for c=32, h=7, w=7 after the two strided convolutions
        self.latent_dim_size = latent_dim_size
        self.hidden_dim_size = hidden_dim_size
        self.encoder = nn.Sequential(
            Conv2d(in_channels=1, out_channels=16, kernel_size=4, stride=2, padding=1),
            ReLU(),
            Conv2d(in_channels=16, out_channels=32, kernel_size=4, stride=2, padding=1),
            ReLU(),
            einops.layers.torch.Rearrange("b c h w -> b (c h w)"), #flatten
            nn.Linear(self.in_channels1, hidden_dim_size),
            ReLU(),
            nn.Linear(hidden_dim_size, latent_dim_size)
        )
        self.decoder = nn.Sequential(
            nn.Linear(in_features=latent_dim_size, out_features=hidden_dim_size),
            ReLU(),
            nn.Linear(in_features=hidden_dim_size, out_features=self.in_channels1),
            einops.layers.torch.Rearrange('b (c h w) -> b c h w', c=32, h=7, w=7),
            nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=4, stride=2, padding=1),
            ReLU(),
            nn.ConvTranspose2d(in_channels=16, out_channels=1, kernel_size=4, stride=2, padding=1)
        )

    def forward(self, x: Tensor) -> Tensor:
        """Returns the reconstruction of the input, after mapping through encoder & decoder."""
        z = self.encoder(x)
        x_reconstructed = self.decoder(z)
        return x_reconstructed


tests.test_autoencoder(Autoencoder)

All tests in `test_autoencoder` passed!


# Training the Autoencoder

Now let's write a training loop which works with mean squared error loss between the original and the reconstructed data.

In [ ]:
@dataclass
class AutoencoderArgs:
    # architecture
    latent_dim_size: int = 5
    hidden_dim_size: int = 128

    # data / training
    dataset: Literal["MNIST", "CELEB"] = "MNIST"
    batch_size: int = 512
    epochs: int = 10
    lr: float = 1e-3
    betas: tuple[float, float] = (0.5, 0.999)

    # logging
    use_wandb: bool = True
    wandb_project: str | None = "day5-autoencoder"
    wandb_name: str | None = None
    log_every_n_steps: int = 250


class AutoencoderTrainer:
    def __init__(self, args: AutoencoderArgs):
        self.args = args
        self.trainset = get_dataset(args.dataset)
        self.trainloader = DataLoader(self.trainset, batch_size=args.batch_size, shuffle=True)
        self.model = Autoencoder(
            latent_dim_size=args.latent_dim_size,
            hidden_dim_size=args.hidden_dim_size,
        ).to(device)
        self.optimizer = t.optim.Adam(self.model.parameters(), lr=args.lr, betas=args.betas)

    def training_step(self, img: Tensor) -> Tensor:
        """
        Performs a training step on the batch of images in `img`. Returns the loss. Logs to wandb
        if enabled.
        """
        raise NotImplementedError()

    @t.inference_mode()
    def log_samples(self) -> None:
        """
        Evaluates model on holdout data, either logging to weights & biases or displaying output.
        """
        assert self.step > 0, "First call should come after a training step. Remember to increment `self.step`."
        output = self.model(HOLDOUT_DATA)
        if self.args.use_wandb:
            output = (output - output.min()) / (output.max() - output.min())  # Normalize to [0, 1]
            output = (output * 255).to(dtype=t.uint8)  # Convert to uint8 for logging
            wandb.log({"images": [wandb.Image(arr) for arr in output.cpu().numpy()]}, step=self.step)
        else:
            display_data(t.concat([HOLDOUT_DATA, output]), nrows=2, title="AE reconstructions")

    def train(self) -> Autoencoder:
        """Performs a full training run."""
        self.step = 0
        if self.args.use_wandb:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name)
            wandb.watch(self.model)

        # YOUR CODE HERE - iterate over epochs, and train your model

        if self.args.use_wandb:
            wandb.finish()

        return self.model


args = AutoencoderArgs(use_wandb=False)
trainer = AutoencoderTrainer(args)
autoencoder = trainer.train()